# Forgetting

Build the forgetting loop: 
- discount the learned curve by $\gamma$ at each step:
    - first toward zero,
    - then toward the prior.
- make the bump vanish mid run
[redacted]

The last unit in the Bayesian Optimization sprint [redacted].

I'm excited

Recall:

- $\mu (x)$ comes from $m = A^{-1}b$
- $\sigma_f(x)$ comes from the uncertainty band $A^{-1}$

- **A** is the posterior precision over the 20 weights: how certain the agent is about the scene's saliency. The "where I've looked" half of the map.
- **b** is the evidence: the running total of what the agent has seen. Each look adds that location's saliency reading, weighted by sensor precision. The "what I saw" half of the map.

### Two ways to forget.

Each look in this notebook runs in this order: 
- read the map and compute $\epsilon^2$
- forget
- update

Toward-zero forgetting:
$$A \leftarrow \gamma A, b \leftarrow \gamma b$$

Toward-prior forgetting:
$$A \leftarrow \gamma A + (1 - \gamma)\alpha I, b \leftarrow \gamma b$$


Bonus (?) covariance forgetting?

### Step 13: One forgetting step, no new data.

Apply each rule once, with $\gamma$ = 0.9, to the five-look map of step 3.

*Predict first: What happens to the mean and to the variance at 0.7, where you looked, and at 0.2, where you did not?*

Prediction answer.
- For toward-zero forgetting:
    - at 0.2 and 0.7, mean is unchanged. The two scalings on $A^{-1}$ and $b$ cancel out. 
    - Variance grows everywhere by 1/09 ~= 1.11. Shrinking the precision $A$ makes uncertainty $A^{-1}$ grow. 
- For toward-prior forgetting:
    - At 0.7 (looked at, A large there), most of its certainty came from data, and 10% of that is now replaced by a small prior term. So the variance grows by about 1/0.9. The mean barely moves: it slips slightly towards 0, because the added prior term pulls a little.
    - At 0.2, there's little data-based certainty to lose. So variance barely moves, and mean stays about 0.

In [ ]:
# Setup, from units 10-11
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True)

centres = np.linspace(0, 1, 20)
width = 0.07
alpha = 1.0
sigma_n = 0.1
lattice = np.linspace(0, 1, 101)

def Phi(x):
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    return np.exp(-(x - centres)**2 / (2 * width**2))

P = Phi(lattice)

def f_true(x):
    return np.exp(-(x - 0.7)**2 / (2 * 0.08**2))

def fresh_map():
    return alpha * np.eye(20), np.zeros(20)

def update(A, b, x, y):
    phi = Phi([x])[0]
    return A + np.outer(phi, phi) / sigma_n**2, b + phi * y / sigma_n**2

def read_lattice(A, b):
    A_inv = np.linalg.inv(A)
    mu = P @ (A_inv @ b)
    sigma_f = np.sqrt(np.sum(P @ A_inv * P, axis=1))
    return mu, sigma_f

def read_map(A, b, x):
    phi = Phi([x])[0]
    A_inv = np.linalg.inv(A)
    return phi @ A_inv @ b, phi @ A_inv @ phi # mu(x), sigma_f^2(x)

In [ ]:
# The two forgetting rules
def forget_zero(A, b, gamma):
    """Toward zero: shrink everything"""
    return gamma * A, gamma * b

def forget_prior(A, b, gamma):
    """Toward the prior: shrink, then top A back up with a little prior"""
    return gamma * A + (1 - gamma) * alpha * np.eye(20), gamma * b

In [ ]:
# Step 13: apply forgetting rules at 0.7, 0.2
# Rebuild step 3's five-look map
x_obs = np.array([0.10, 0.35, 0.60, 0.70, 0.90])
y_obs = np.array([0.02, -0.05, 0.41, 0.93, 0.04])
A5, b5 = fresh_map()
for x, y in zip(x_obs, y_obs):
    A5, b5 = update(A5, b5, x, y)

gamma = 0.9
maps = {
    'before':       (A5, b5),
    'toward zero':  forget_zero(A5, b5, gamma),
    'toward prior': forget_prior(A5, b5, gamma)
}

for x in [0.7, 0.2]:
    print(f'--- x = {x} ---')
    _, var_before = read_map(A5, b5, x)
    for name, (A_, b_) in maps.items():
        mu_x, var_x = read_map(A_, b_, x)
        print(f'{name:13s} mean = {mu_x:+.4f}  variance = {var_x:.4f}  '
              f'(x{var_x / var_before:.3f} vs before)')
        
# Plot all three maps
for name, (A_, b_) in maps.items():
    mu, sf = read_lattice(A_, b_)
    line, = plt.plot(lattice, mu, label=name)
    plt.fill_between(lattice, mu - 2*sf, mu + 2*sf, color=line.get_color(), alpha=0.1)
plt.scatter(x_obs, y_obs, c='k', zorder=3, label='gazes')
plt.ylim(-3.5, 3.5)
plt.xlabel('x (location in scene)'); plt.legend();

Thesis takeaway: forgetting towards the prior only erodes what was *learned*. Where the agent never knew stays at "prior ignorance". This becomes more dramatic in step 14.

### Step 14: Stare test

Fresh map, $\gamma = 0.98$. Look at 0.7 fifty times, with $y = 1$ and no noise.

*Predict first: $\sigma_f$ at 0.2 under each rule. Then, the same after 500 looks.*

Prediction answer.
- Looking at 0.7 never touches 0.2. So the question is "what does each *forgetting rule* do to an untouched part of the map over 50 looks?"
- After 50 looks at 0.7, $\sigma_f(0.2)$
    - No forgetting: nothing changes. Uncertainty at 0.2 stays at the prior band, ~1.54 (step 2)
    - Toward zero: each look multiples the precision by 0.98 -- $0.98^{50}$ ~= 0.36. So the variance is about 1/0.36 ~= 2.7x the prior variance, and $sigma_f$ grows by about $\sqrt 2.7$ ~= 1.65x. So it grows to about 2.5. The agent becomes *more unsure* about 0.2 by about 1.65x.
    - Toward prior: near 0.2, A is still the prior ($\alpha I$), and the rule gives $0.98 \alpha I + 0.02 \alpha I = \alpha I$, so it is unchanged.
- After 500 looks
    - Toward zero: precision $0.98^{500}$ ~= 0.00004 of the prior's, so variance is about 24,000x and $\sigma_f$ about 155x. That is, roughly 240, and still growing without limit.
    - No forgetting and toward prior: still unchanged, about 1.54.

In [ ]:
# Simulate the 50, 500 looks
def stare(rule, n_looks, gamma=0.98, x_look=0.7, y_look=1.0):
    """Fresh map; look at x_look n_looks times (no noise). rule=None means no forgetting.
    Returns sigma_f at 0.2 and at 0.7 after each look."""
    A, b = fresh_map()
    sd_02, sd_07 = [], []
    for t in range(n_looks):
        if rule is not None:
            A, b = rule(A, b, gamma)        # forget...
        A, b = update(A, b, x_look, y_look) # ...then update (no noise: y is exactly 1)
        sd_02.append(np.sqrt(read_map(A, b, 0.2)[1]))
        sd_07.append(np.sqrt(read_map(A, b, 0.7)[1]))
    return np.array(sd_02), np.array(sd_07)

rules = {'no forgetting': None, 'toward zero': forget_zero, 'toward prior': forget_prior}

for n in [50, 500]:
    print(f'--- after {n} looks at 0.7 ---')
    for name, rule in rules.items():
        sd_02, _ = stare(rule, n)
        print(f'{name:14s} sigma_f(0.2) = {sd_02[-1]:.4f}')
        
# How sigma_f(0.2) evolves over the 50 looks
for name, rule in rules.items():
    sd_02, _ = stare(rule, 50)
    plt.plot(range(1, 51), sd_02, label=name)
    plt.xlabel('look number'); plt.ylabel('σ_f at 0.2 (never looked at)'); plt.legend();

Thesis takeaway: under toward-zero forgetting, any place the agent hasn't looked lately becomes extremely uncertain, and UCB ($\mu + \kappa \sigma_f$) would be pulled there just for being unvisited. Toward-prior forgetting doesn't have this problem: unvisited places just sit at "prior ignorance". 

### Step 15: Memory length

Same run, but read $\sigma_f$ at 0.7 itself.

*Predict first: Without forgetting the band shrinks like $\sigma_n/\sqrt n$*. Where does it stop when $\gamma = 0.98$?

Prediction answer.
- Without forgetting, every look counts fully. So it keeps shrinking towards 0 forever.
- With forgetting ($\gamma = 0.98$), each look counts a bit less every time a new one arrives. The newest look counts 1, the one before 0.98, then $0.98^2$, etc. So how many looks does the agent effectively remember? Add up the weights:
$$1 + \gamma + \gamma^2 + \dots = \frac{1}{1-\gamma} = \frac{1}{0.02} = 50 \text{looks}$$
- That's the memory length. However long it stares, the agent never has more than about 50 looks' worth of evidence, so the band stops shrinking at about
$$\sigma_f \approx \frac{\sigma_n}{\sqrt {50}} = \sigma_n \sqrt{1-\gamma} \approx 0.014$$
- After 50 looks, the memory isn't full yet: the effective count is $(1 - 0.98^{50})/0.02 \approx 32$ looks, so $\sigma_f \approx 0.1/\sqrt 32 \approx 0.018$, a bit wider than without forgetting.
- Both rules give the same answer at 0.7. The prior's small top-up is negligible over 50 looks.

In [ ]:
# Read out the stare for σ_f(0.7) for 50, 500 looks
gamma = 0.98
floor = sigma_n * np.sqrt(1 - gamma) # long-run limit with forgetting

for n in [50, 500]:
    print(f'--- after {n} looks at 0.7 ---')
    eff_n = (1 - gamma**n) / (1 - gamma) # effective number of looks remembered
    print(f'  predictions: no forgetting {sigma_n/np.sqrt(n):.4f}, '
          f'forgetting {sigma_n/np.sqrt(eff_n):.4f} (eff. n = {eff_n:.1f})')
    for name, rule in rules.items():
        _, sd_07 = stare(rule, n)
        print(f'  {name:14s} sigma_f(0.7) = {sd_07[-1]:.4f}')
print(f'long-run floor sigma_n * sqrt(1 - gamma) = {floor:.4f}, memory = {1/(1-gamma):.0f} looks')

# The band at 0.7 over 5000 looks
n_long = 500
looks = np.arange(1, n_long + 1)
for name, rule in rules.items():
    _, sd_07 = stare(rule, n_long)
    plt.plot(looks, sd_07, label=name)
plt.plot(looks, sigma_n / np.sqrt(looks), 'k:', label='σ_n/√(n)')
plt.axhline(floor, color='gray', ls='--', label='σ_n √(1-γ) floor')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('look number'); plt.ylabel('σ_f at 0.7 (where it stares)'); plt.legend();

Thesis takeaway: $\gamma$ is a **memory length**: $\gamma = 0.98$ means "trust about the last 50 looks". The agent's certainty has a floor. It can never become so sure that it stops being able to notice change. That's the point of forgetting, and it has a price (which step 17 measures).

### Step 16: The vanishing bump

Sweep lattice indices 0, 5, ..., 100 in order: 21 looks a sweep, no noise. The bump is there for sweeps 1 to 10 and gone (y=0 everywhere) for sweeps 11 to 20. Forget towards prior. Record $\mu(0.7)$ after every sweep, for $\gamma = 1, 0.99, 0.98, 0.95$

*Predict first: For each $\gamma$, the number of sweeps before $\mu(0.7)$ falls below 0.5.*

Prediction answer.
- abcabc